In [ ]:
import os
import shutil
import time
import base64
import requests
from urllib.parse import quote

# Selenium 관련 라이브러리
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 크롤링 제한 우회
import undetected_chromedriver as uc
import time

# 사진 크기 필터링
from io import BytesIO
from PIL import Image

start = time.time()

# ==========================================
# 사용자 설정 및 경로
# ==========================================
BASE_PATH = 'C:\\Users\\1004t\\Desktop\\중앙대'
QUERY_FILE_PATH = os.path.join(BASE_PATH, '한국 연예인1.txt')

LIMITS = {
    "정면": 40,      # 검색어 + 정면 20장
    "옆모습": 30     # 검색어 + 옆모습 20장
}

# 💡 최소 이미지 크기 설정 (이 값보다 작으면 필터링)
MIN_WIDTH = 200
MIN_HEIGHT = 200


# 2. 텍스트 파일에서 쿼리 읽어오기
if not os.path.exists(QUERY_FILE_PATH):
    os.makedirs(BASE_PATH, exist_ok=True)
    with open(QUERY_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("수지\n아이유\n")
    print(f"[{QUERY_FILE_PATH}] 예시 파일이 생성되었습니다.")

with open(QUERY_FILE_PATH, 'r', encoding='utf-8') as f:
    queries = [line.strip() for line in f if line.strip()]

print(f"불러온 쿼리 목록: {queries}\n")

# 실습용 타겟 지정 (전체 실행 시 주석 처리)
queries = ["손석구"]

# ==========================================
# 3. Selenium 웹드라이버 설정 (Colab 전용 Headless)
# ==========================================
chrome_options = Options()
# chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
# Colab 내장 크롬 드라이버 경로
driver = uc.Chrome(options=chrome_options, version_main=145)

# ==========================================
# 4. 크롤링 및 다운로드 실행
# ==========================================
for query in queries:
    query_start = time.time()
    query_dir = os.path.join(BASE_PATH, "데이터셋", "한국 연예인1", query)
    print(f"\n========================================\n대상: {query}\n저장 경로: {query_dir}\n========================================")

    # 안전하게 기존 폴더 삭제 후 재생성
    if os.path.exists(query_dir):
        shutil.rmtree(query_dir)
    os.makedirs(query_dir, exist_ok=True)

    search_types = ["정면", "옆모습"]

    for s_type in search_types:
#         search_query = f"{query}" if s_type == "얼굴" else f"{query} {s_type}"
        search_query = f"{query} {s_type} 고화질"
        limit_count = LIMITS[s_type]

        print(f"\n>>> '{search_query}' 검색 및 스크롤 시작... (목표: {limit_count}장)")

        # 구글 이미지 검색 URL 접속
        url = f"https://www.google.com/search?q={quote(search_query)}&tbm=isch"
        driver.get(url)

        try:
            # 1단계: 브라우저 자체가 "나 로딩 다 했어(complete)"라고 신호를 보낼 때까지 최대 15초 대기
            WebDriverWait(driver, 15).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )

            # 2단계: 우리가 크롤링할 구글 이미지 썸네일('.YQ4gaf')이 화면에 나타날 때까지 최대 15초 대기
            # 만약 1.5초 만에 이미지가 나타나면 15초를 채우지 않고 즉시 다음 줄로 넘어갑니다! (시간 엄청 절약됨)
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".YQ4gaf"))
            )
            print("✔️ 초기 이미지 로딩 완료!")

        except Exception as e:
            print(f"⚠️ 페이지 로딩 타임아웃! 인터넷 연결을 확인하세요.")
            continue # 로딩 실패 시 에러 뿜지 않고 다음 검색어(옆모습 등)로 부드럽게 넘어감

        # 동적 페이지 스크롤 (이미지를 충분히 불러오기 위해)
        last_height = driver.execute_script("return document.body.scrollHeight")
        for _ in range(5): # 목표 개수에 따라 스크롤 횟수 조절 가능
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(7)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        # 구글 이미지의 썸네일 클래스인 'rg_i'를 찾아 추출
        image_elements = driver.find_elements(By.CSS_SELECTOR, ".YQ4gaf")

        saved_count = 1
        for img in image_elements:
            if saved_count > limit_count:
                break

            try:
                # 이미지 소스 URL 가져오기 (src가 없으면 data-src 확인)
                src = img.get_attribute('src')
                if not src:
                    src = img.get_attribute('data-src')

                if not src:
                    continue

                # 우리가 원하는 파일명으로 즉시 저장
                # 확장자는 기본적으로 jpg로 통일하여 안전하게 저장합니다.
                file_name = f"{query}_{s_type}_{saved_count}.jpg"
                file_path = os.path.join(query_dir, file_name)

                img_data = None

                # 1. Base64 이미지 데이터 추출
                if src.startswith('data:image'):
                    base64_data = src.split(',')[1]
                    img_data = base64.b64decode(base64_data)

                # 2. 일반 HTTP URL 이미지 데이터 추출
                elif src.startswith('http'):
                    response = requests.get(src, timeout=10)
                    if response.status_code == 200:
                        img_data = response.content

                # 💡 [핵심] 이미지 크기 검사 로직 추가
                if img_data:
                    # 메모리 내에서 바이너리 데이터를 이미지 객체로 전환 (디스크 저장 전 검사)
                    with Image.open(BytesIO(img_data)) as pillow_img:
                        width, height = pillow_img.size

                        # 둘 중 하나라도 설정해둔 해상도 이상이면 통과
                        if width < MIN_WIDTH and height < MIN_HEIGHT:
                            # 디버깅 편의를 위해 로그 출력
                            # print(f"⏩ 너무 작은 이미지 스킵 ({width}x{height})")
                            continue

                    # 크기 통과 시 최종 저장
                    with open(file_path, 'wb') as f:
                        f.write(img_data)
                        saved_count += 1

            except Exception as e:
                # 다운로드 중 에러 발생 시 무시하고 다음 이미지로 진행
                print(e)
                continue

        print(f"     └─ {s_type} 사진 {saved_count - 1}장 저장 완료!")
    print(f"{query} : {(time.time()-query_start)} 초")

print("\n크롤링이 모두 완료되었습니다! 브라우저를 종료합니다.")
print(f"{(time.time() - start) / 60} 분")
driver.quit()